# Mantis Vision — Train in Google Colab

Run these cells top to bottom. Before starting: **Runtime -> Change runtime type -> GPU**.

This notebook: clones the repo, installs dependencies, pulls the labeled dataset down from Kaggle, trains the EfficientNet-B0 health classifier, evaluates it, and lets you download the resulting checkpoint.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/Arran0/MantisVision.git
%cd MantisVision/ml

## 2. Install dependencies

Colab already has torch/torchvision preinstalled with GPU support, so this just adds the extra packages the pipeline needs.

In [ ]:
!pip install -q grad-cam kaggle fastapi python-multipart

## 3. Kaggle credentials

Go to kaggle.com/settings -> API -> "Create New Token" and copy the **username** and **key** shown there (Kaggle no longer gives you a downloadable `kaggle.json` by default — just a token to copy). Run the cell below, then paste your username when prompted and your key into the hidden password box.

In [ ]:
import os
from getpass import getpass

os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
os.environ["KAGGLE_KEY"] = getpass("Kaggle API key: ").strip()

## 4. Download the dataset from Kaggle

Replace `YOUR_USERNAME/YOUR_DATASET_SLUG` with the id from your Kaggle dataset's URL (`kaggle.com/datasets/<that part>`).

In [ ]:
DATASET_ID = "YOUR_USERNAME/YOUR_DATASET_SLUG"  # <-- edit this
!python scripts/kaggle_sync.py download --dataset {DATASET_ID}

## 4b. If validate_dataset reports 0 images everywhere, run this

Common cause: Kaggle stored your upload as a single un-extracted `.zip` file
instead of real folders, or there's an extra wrapper folder around
`train/validation/test`. This cell shows you exactly what got downloaded and
automatically fixes both cases in place (no need to re-upload to Kaggle).

In [ ]:
import pathlib
import shutil
import zipfile

dataset_dir = pathlib.Path("dataset")
expected_splits = {"train", "validation", "test"}


def print_tree(path, prefix="", depth=0, max_depth=3):
    if depth > max_depth:
        return
    for entry in sorted(path.iterdir()):
        print(f"{prefix}{entry.name}{'/' if entry.is_dir() else ''}")
        if entry.is_dir():
            print_tree(entry, prefix + "  ", depth + 1, max_depth)


print(f"Contents of {dataset_dir.resolve()} before fixing:\n")
print_tree(dataset_dir)

# Case 1: Kaggle stored the upload as a plain .zip instead of extracting it.
stray_zips = list(dataset_dir.rglob("*.zip"))
for z in stray_zips:
    print(f"\nExtracting leftover archive: {z}")
    with zipfile.ZipFile(z) as zf:
        zf.extractall(z.parent)
    z.unlink()

# Case 2: train/validation/test are nested one level deeper inside a wrapper
# folder (e.g. the zip was created from the parent folder, not its contents).
top_level_dirs = {p.name for p in dataset_dir.iterdir() if p.is_dir()}
if not expected_splits.issubset(top_level_dirs):
    for child in dataset_dir.iterdir():
        if not child.is_dir():
            continue
        grandchildren = {p.name for p in child.iterdir() if p.is_dir()}
        if expected_splits.issubset(grandchildren):
            print(f"\nFound train/validation/test nested inside '{child.name}/', flattening...")
            for item in child.iterdir():
                shutil.move(str(item), str(dataset_dir / item.name))
            child.rmdir()
            break

print(f"\nContents of {dataset_dir.resolve()} after fixing:\n")
print_tree(dataset_dir)

if not expected_splits.issubset({p.name for p in dataset_dir.iterdir() if p.is_dir()}):
    print(
        "\nStill missing train/validation/test at the top level. Open your dataset's "
        "page on kaggle.com and check the file listing there directly — if it shows "
        "something other than train/validation/test folders, the fix has to happen by "
        "re-uploading (drag the train/validation/test folders themselves into the Kaggle "
        "uploader, not a zip of their parent folder)."
    )
else:
    print("\nLooks correct now — re-run the validate_dataset cell below.")


## 5. Validate the dataset

Checks every class folder exists, every image opens correctly, and flags underrepresented classes.

In [ ]:
!python -m src.data.validate_dataset

## 6. Train

Two-phase schedule (frozen backbone, then fine-tune) with early stopping. Saves the best checkpoint to `ml/checkpoints/best_model.pt`. This can take a while depending on dataset size — Colab's free GPU tier helps a lot here.

In [ ]:
!python -m src.train

## 7. Evaluate

Accuracy/precision/recall/F1 (macro + per-class), confusion matrix, ROC AUC on the held-out test split.

In [ ]:
!python -m src.evaluate

from IPython.display import Image, display
display(Image(filename="reports/confusion_matrix.png"))

## 8. Download the trained checkpoint

Colab runtimes are ephemeral — grab the checkpoint now so you don't lose it when the session disconnects. You'll need this file to run the inference API later.

In [ ]:
from google.colab import files
files.download("checkpoints/best_model.pt")

## Optional: try a Grad-CAM explanation on one photo

Upload a single test photo and see the heatmap of what the model looked at.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick one photo
photo_path = list(uploaded.keys())[0]
!python -m src.gradcam "{photo_path}"

from IPython.display import Image, display
import pathlib
display(Image(filename=str(pathlib.Path(photo_path).with_suffix('.gradcam.png'))))